# Build to Verify
## Four-Bar Linkage Design for the MiniClaw Jaw (One Side)
## ME 493B — AI in Product Development | Mini-Project 4, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Friday, May 29, 2026 at 11:59 PM
**Time estimate:** 3–4 hours of focused work
**Points:** 50 (Part A). Part B is worth 50 points separately and is
your team's integration of the individual four-bar designs into one
prototype-ready MiniClaw.

---

### What changed

MP1 was *define the problem.* MP2 was *research the problem.* MP3 was
*productize your engineering work.* MP4 Part A is **use the productized
stack to design a real subsystem and verify it holds up**, before team
integration in Part B.

You arrive at MP4 with a working stack from MP3: skills, an MCP server
exposing your MiniClaw RAG, a configured host (Copilot agent mode /
Claude Desktop / Cursor). This notebook does not re-teach the stack.
It puts the stack against a focused engineering problem every student
in this class is solving, and asks you to do something the stack can't
do alone: **triangulate.**

### The problem

Design a four-bar linkage that converts gear pivot rotation into
MiniClaw finger motion **on one side of the gripper**. You assume:

- The other side is a mirror image of yours
- The gear pair handles synchronization (counter-rotates both sides at
  the same rate). Designing the gear pair is your team's Part B work.
- Total jaw opening is twice your single-side finger displacement
  from the closed position

Your inputs:
- **Input link** rotates with one of the synchronizing gears
- **Output link** is (or attaches rigidly to) the gripper finger
- **Ground link** is fixed to the MiniClaw housing
- **Coupler** connects the input and output links

Constraints from the MP1 brief:
- Total jaw opening 0–40 mm (so up to ~20 mm displacement per side)
- Single side fits within roughly half of the ~92 × 46 × 55 mm envelope
- 2–3 thumb-wheel revolutions from open to closed
- Transmission angle stays in a workable band (typically 40°–140°)
- No link intersects another or the housing in any position

Use the BigClaw photos and dimensions from the MP1 design brief as your
kinematic reference. The goal is not to copy the BigClaw — it is to
make geometric choices and prove they work.

### The triangulation principle

A single AI answer is not evidence. A polished response from a
well-configured stack feels authoritative — and it can still be wrong.
The only honest way to trust an engineering answer is to triangulate.

For your linkage, you produce three independent paths:

1. **Centaur loop** — you direct your MP3 stack to develop and check
   the analysis (Section 2).
2. **Simulation or visualization** — you use a tool (Rapier.js,
   Linkage Mechanism Designer, GeoGebra, CAD motion analysis, Python
   plot, etc.) to produce visible motion (Section 5).
3. **Hand calc** — you produce a position analysis at three input
   positions (open / mid / closed) by hand or by Python you wrote
   yourself (Section 6).

Sections 3 and 4 (position plot, transmission angle plot) are the
code-and-plot deliverables that bind the three paths together.

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|------------------------|
| 1. Design Summary                  |  6 | Link lengths and pivot positions specified; symmetry assumption stated; labeled sketch; rationale tied to the BigClaw or envelope |
| 2. Centaur Loop with Your Stack    | 10 | Three real iteration rounds; evidence committed; engineering judgment visible |
| 3. Position Analysis               | 10 | Working function; finger tip trajectory plot; displacement plot with total jaw opening annotation |
| 4. Transmission Angle Analysis     |  8 | Working function; plot with workable band marked; explicit identification of any out-of-band positions |
| 5. Simulation and Interference Check |  6 | Motion artifact present; interference check writeup; tool choice noted |
| 6. Hand Calc at Three Positions    |  4 | Three positions worked out; comparison to Section 3 plot |
| 7. Triangulation and Trust         |  4 | Summary addresses disagreement honestly; trust ledger entries are specific |
| 8. Reflection                      |  2 | Thoughtful 3–4 sentence reflection |
| **Total** | **50** | |

### What this notebook is NOT

- Not a re-teach of MP3. Use what works in your stack; document gaps.
- Not a velocity / mechanical-advantage analysis. Optional stretch.
- Not a Grashof condition exercise. Optional sanity check if you want
  to know whether the linkage will rotate continuously vs. rock
  through a limited range; not required for full marks.
- Not the gear pair design. That is your team's Part B work.

---
## Section 0: Setup

Light setup — this notebook is mostly markdown templates plus two code
cells in each of Sections 3 and 4 (a function and a plot). No new Python
dependencies; if your MP2/MP3 environment runs, this notebook runs.

**Where the artifacts live:**

```
MP4/Part A/
├── MP4_PartA_Build_to_Verify.ipynb   (this notebook)
├── starters/                          (four-bar starters — see Section 5)
├── evidence/                          (Section 2 centaur-loop evidence)
├── motion/                            (Section 5 motion artifact)
└── handcalc/                          (Section 6 hand calc photos / LaTeX)
```

Run the next cell to confirm the environment.

In [ ]:
# Pre-written setup cell (do not modify).
import json
import os
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# scipy.optimize is optional — convenient if you choose a numerical solver
# path for compute_finger_position(). The analytical (two-circle
# intersection) path uses only numpy.
try:
    from scipy import optimize as _opt
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False

# Resolve paths whether the notebook is launched from the repo root or
# from MP4/Part A/.
HERE = Path.cwd()
if (HERE / "MP4" / "Part A").exists():
    BASE = HERE / "MP4" / "Part A"
elif HERE.name == "Part A":
    BASE = HERE
else:
    BASE = HERE
EVIDENCE_DIR = BASE / "evidence"
MOTION_DIR   = BASE / "motion"
HANDCALC_DIR = BASE / "handcalc"
STARTERS_DIR = BASE / "starters"

for d in (EVIDENCE_DIR, MOTION_DIR, HANDCALC_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Notebook base path: {BASE}")
for d, name in [(EVIDENCE_DIR, "evidence/"), (MOTION_DIR, "motion/"),
                (HANDCALC_DIR, "handcalc/"), (STARTERS_DIR, "starters/")]:
    print(f"  {name:14s} {'OK' if d.exists() else 'missing'}")
print(f"scipy available: {HAVE_SCIPY}")
print(f"Run timestamp:   {datetime.now().isoformat(timespec='seconds')}")

---
## Section 1: Design Summary [6 pts]

State your linkage geometry up front. Everything in Sections 3–7 references
these numbers. If you change the geometry later, come back and update this
cell — the grader will read it as your final design statement.

Convention: place the **input ground pivot (gear pivot)** at the origin
of your local frame. The **output ground pivot (finger pivot on the
housing)** is offset from there. All distances in millimetres, angles in
degrees.

### ✏️ YOUR TURN — Fill this in

## My Four-Bar Linkage Design — One Side, with Symmetry Assumption

**Architecture I'm assuming:** Two-layer — gear pair synchronizes the two
sides (counter-rotate at the same rate); each side has its own four-bar
that shapes finger motion. I am designing one side. The other side is a
mirror image of this one. Total jaw opening is twice my single-side
finger displacement from the closed position.

**Link lengths (mm):**
- Ground (L1): _..._
- Input (L2):  _..._
- Coupler (L3): _..._
- Output (L4): _..._

**Pivot positions (mm), in the local frame where the input ground pivot is at the origin:**
- Input ground pivot (gear pivot): (0, 0)
- Output ground pivot (finger pivot on housing): ( _..._ , _..._ )

**Tip extension past joint B along the output link (mm):** _..._
*(this is what makes "finger length" different from L4 — it's the part of
the finger past the moving joint; can be 0)*

**Input range:** from _..._° to _..._°

**Target single-side finger displacement (mm):** _..._
*(so total jaw opening = 2 × this)*

**Labeled sketch or screenshot:** _embed image or link below_
`<!-- ![Linkage sketch](evidence/linkage_sketch.png) -->`

**Design rationale (3–4 sentences):**
> _Why these choices? What in the BigClaw reference (or in the MP1
> envelope constraints) led to these specific lengths and positions?_

---
## Section 2: Centaur Loop with Your Stack [10 pts]

Three iteration rounds with your MP3 stack on your linkage design. Each
round captures five things:

1. **What you asked the AI** (the prompt or interaction)
2. **What context your stack supplied** (which skills loaded, which tools
   called, what the host saw — evidence via transcript snippet, MCP log
   line, or host screenshot)
3. **What the AI produced** (the analysis, derivation, code, or
   recommendation)
4. **Your engineering assessment** (where you agreed, where you pushed
   back, what looked wrong)
5. **What changed in the next round**

**Useful loops for this design problem.** You're not limited to these,
but they tend to land:

- Ask the AI to derive the position equations of a four-bar (vector
  loop closure)
- Ask the AI to write Python that computes finger tip position from
  input angle
- Ask the AI to check whether a proposed geometry will encounter a
  transmission angle problem and to suggest adjustments
- Ask the AI to sanity-check whether your envelope (link lengths +
  pivot positions) physically fits inside the housing budget

**Evidence rule.** For each round, commit at least one artifact to
`MP4/Part A/evidence/`:

- A screenshot of the host UI showing the interaction (`.png`)
- A transcript snippet pasted into a `.md` file
- An MCP server log line (paste into a `.md`, or commit the log)

The grader looks at this folder. If there's no evidence, the round
earns at most 50% of its points.

**Stack continuity.** Use what works. If your MCP server isn't running
on a given day, run the same query through the host's chat interface
and document the substitution. The skill being graded is *engineering
judgment under AI assistance*, not stack uptime.

### Round 1 — Initial framing

**Date / time:** _yyyy-mm-dd HH:MM_

**Host & stack used:** _e.g., "Copilot agent mode in VS Code, MCP server
on stdio, miniclaw_skill loaded as `.github/copilot-instructions.md`."_

**What I asked the AI:**
> _Paste your prompt or describe the interaction._

**What context my stack supplied:**
> _Which skill(s) were active, which tool(s) the AI called, what the
> RAG returned. Cite your evidence file: `evidence/round1_<short>.png`
> or `.md`._

**What the AI produced:**
> _Summarize the answer in 2–4 sentences. Don't paste the whole reply
> unless a specific phrase matters._

**My engineering assessment:**
> _Where did the AI nail it? Where was it wrong, hand-wavy, or
> over-confident? Be specific._

**What changed for round 2:**
> _The prompt, the context, the framing — what did you adjust based on
> what came back?_

---
### Round 2 — Refinement

**Date / time:** _..._
**Host & stack used:** _..._
**What I asked the AI:**
> _..._
**What context my stack supplied:**
> _..._  Evidence: `evidence/round2_<short>.png` or `.md`
**What the AI produced:**
> _..._
**My engineering assessment:**
> _..._
**What changed for round 3:**
> _..._

---
### Round 3 — Convergence (or divergence)

**Date / time:** _..._
**Host & stack used:** _..._
**What I asked the AI:**
> _..._
**What context my stack supplied:**
> _..._  Evidence: `evidence/round3_<short>.png` or `.md`
**What the AI produced:**
> _..._
**My engineering assessment:**
> _..._
**Stack notes:**
> _One short paragraph — what parts of your MP3 stack did you actually
> use for these three rounds, and what didn't work? This goes into your
> Part B trust assessment too._

In [ ]:
# Quick check — list the evidence files you've committed for Section 2.
# The grader will look here. If it's empty, your centaur log isn't
# backed by evidence yet.
print("Evidence committed for Section 2:")
found = sorted(p for p in EVIDENCE_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — drop screenshots / transcript snippets into evidence/)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 3: Position Analysis [10 pts]

The first of three required analyses. Compute and plot:

1. **Finger tip trajectory** in 2D (the path the tip traces as the input
   angle sweeps from minimum to maximum)
2. **Single-side finger displacement** vs. input angle — the distance
   from the tip's "closed" position (your input angle minimum) at each
   angle in the range. Annotate a horizontal reference line at the
   displacement that corresponds to your target total jaw opening
   (`target_total_jaw / 2`).

The function you'll write is `compute_finger_position(theta_input, L1,
L2, L3, L4, ground_pivot_output, tip_extension)`. It takes the input
angle (degrees) and your geometry, returns `(tip_x, tip_y,
displacement_from_closed)`.

**Two reasonable approaches:**

- **Analytical (recommended).** Vector loop closure + two-circle
  intersection. Closed form, no solver needed. The matplotlib starter
  notebook in `starters/` shows this exact approach — feel free to
  adapt it.
- **Numerical.** Use `scipy.optimize.fsolve` on the two loop-closure
  equations. More code, but a fine learning exercise if you want it.

Either way — your AI stack is exactly the right tool for help here. Loop
1 of Section 2 is a good place to ask the AI to derive the equations.

In [ ]:
# Step 1 — Implement compute_finger_position().
#
# Convention used by the rest of the notebook:
#   - Input ground pivot O2 is at the origin (0, 0).
#   - Output ground pivot O4 = ground_pivot_output, a tuple (OX, OY) in mm.
#   - The "closed" reference is the tip position at theta_input = THETA_MIN
#     (your minimum input angle from Section 1).
#   - WHERE TO ATTACH THE FINGER: For a parallel-gripper (parallelogram)
#     design where you want the finger to translate without rotating, the
#     tip must be a rigid extension of the COUPLER L3 — extend by
#     `tip_extension` past joint B in the coupler direction (from A toward
#     B). For a parallelogram this direction is FIXED in the world frame
#     (= O4 - O2 direction), so the finger never rotates.
#     **Pitfall:** the older starter put the tip along the OUTPUT crank
#     (direction from O4 to B). That makes the finger swing in a big arc
#     with the output crank — wrong for a parallel gripper. Watch for it.
#     Tip extension can be 0 (finger ends at joint B).
#
# Required return: (tip_x, tip_y, displacement_from_closed)
#   - tip_x, tip_y in mm  (tip position in your local frame)
#   - displacement_from_closed in mm (distance from the tip at THETA_MIN)
#
# The displacement_from_closed at theta_input = THETA_MIN should be 0
# by construction. Pass `closed_tip=None` for the first call to
# initialize it from THETA_MIN; subsequent calls use that fixed reference.
#
# **Two-circle intersection has two assembly modes** (branches). If your
# output looks inside-out — e.g., joint B comes out on the wrong side of
# the A-to-O4 line — you have the wrong branch. Flip the sign of the
# perpendicular term. The link-length check |A-B| = L3 and |O4-B| = L4
# always passes for either branch; it does NOT catch a wrong-branch bug.
# Verify the parallelogram identity B = A + (O4 - O2) if you suspect this.
#
# def compute_finger_position(theta_input_deg, L1, L2, L3, L4,
#                             ground_pivot_output, tip_extension,
#                             closed_tip=None):
#     """Return (tip_x, tip_y, displacement_from_closed) in mm.
#
#     If closed_tip is None, displacement_from_closed is set to 0.0 and
#     the caller should compute it externally for sweep plots.
#     """
#     # YOUR CODE HERE
#     pass

In [ ]:
# Step 2 — Plot the finger tip trajectory and the displacement curve.
#
# 1. Sweep theta_input from THETA_MIN to THETA_MAX.
# 2. Call compute_finger_position() at each angle.
# 3. First plot:  tip_x vs. tip_y  (the trajectory in 2D, equal aspect)
# 4. Second plot: theta_input vs. displacement_from_closed
#                 — annotate horizontal line at TARGET_TOTAL_JAW / 2
#                 — secondary axis or annotation for total jaw opening
#                   (= 2 × displacement)
#
# ---- EDIT TO MATCH YOUR DESIGN (these should equal Section 1) ----
L1 = 46.0
L2 = 14.0
L3 = 44.0
L4 = 22.0
GROUND_PIVOT_OUTPUT = (46.0, 0.0)   # (OX, OY) in mm
TIP_EXTENSION = 18.0                 # mm past joint B along output link
THETA_MIN, THETA_MAX = -60.0, 60.0   # input range, degrees
TARGET_TOTAL_JAW = 40.0              # mm — from the MP1 brief
# ------------------------------------------------------------------

# thetas = np.linspace(THETA_MIN, THETA_MAX, 181)
# # First call to set the closed reference:
# closed = compute_finger_position(THETA_MIN, L1, L2, L3, L4,
#                                  GROUND_PIVOT_OUTPUT, TIP_EXTENSION)
# closed_tip = (closed[0], closed[1])
#
# tips_x, tips_y, disp = [], [], []
# for t in thetas:
#     state = compute_finger_position(t, L1, L2, L3, L4,
#                                     GROUND_PIVOT_OUTPUT, TIP_EXTENSION,
#                                     closed_tip=closed_tip)
#     tips_x.append(state[0]); tips_y.append(state[1]); disp.append(state[2])
#
# fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
# ax1, ax2 = axes
# ax1.plot(tips_x, tips_y, "-")
# ax1.set_aspect("equal"); ax1.grid(alpha=0.3)
# ax1.set_xlabel("tip x (mm)"); ax1.set_ylabel("tip y (mm)")
# ax1.set_title("Finger tip trajectory")
#
# ax2.plot(thetas, disp, "-")
# ax2.axhline(TARGET_TOTAL_JAW / 2, ls="--", color="orange",
#             label=f"target single-side displacement = {TARGET_TOTAL_JAW/2:.1f} mm")
# ax2.set_xlabel("input angle θ_in (deg)")
# ax2.set_ylabel("single-side displacement from closed (mm)")
# # Secondary y-axis showing total jaw opening
# ax2b = ax2.twinx()
# ax2b.set_ylim(2 * np.array(ax2.get_ylim()))
# ax2b.set_ylabel("implied total jaw opening (mm)  [= 2 × disp.]")
# ax2.set_title("Displacement vs. input angle")
# ax2.legend(loc="best", fontsize=9)
# plt.tight_layout(); plt.show()

---
## Section 4: Transmission Angle Analysis [8 pts]

The second required analysis. The transmission angle μ is the angle
between the **coupler L3** and the **output L4** at their shared joint
B. It tells you how efficiently rotational input is converted to motion
at the output:

- μ near 90° → near-ideal force/motion transmission
- μ near 0° or 180° → singularity; the linkage *locks* or jams
- **Workable band:** typically 40°–140°. Outside this band, the linkage
  transmits poorly and may bind under realistic load.

Implement `compute_transmission_angle(theta_input, L1, L2, L3, L4,
ground_pivot_output)` and plot μ across your full input range. Mark the
workable band on the plot. Then write a 1–2 sentence note: *"is my
linkage in the band the whole time? If not, where does it leave?"*

In [ ]:
# Step 1 — Implement compute_transmission_angle().
#
# Approach (one of many):
#   1. Re-use compute_finger_position() to get joint B at each input angle.
#   2. Compute the angle at B between vectors B->A and B->O4.
#   3. Return the absolute angle in degrees (between 0 and 180).
#
# def compute_transmission_angle(theta_input_deg, L1, L2, L3, L4,
#                                ground_pivot_output):
#     """Return the transmission angle at B in degrees (0..180)."""
#     # YOUR CODE HERE
#     pass

In [ ]:
# Step 2 — Plot transmission angle across the input range with the
# workable 40°–140° band marked. Same THETA_MIN / THETA_MAX as Section 3.
#
# WORKABLE_BAND = (40.0, 140.0)
#
# mus = [compute_transmission_angle(t, L1, L2, L3, L4, GROUND_PIVOT_OUTPUT)
#        for t in thetas]
#
# fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(thetas, mus, "-")
# ax.axhspan(*WORKABLE_BAND, color="green", alpha=0.10, label="workable band 40°–140°")
# ax.axhline(WORKABLE_BAND[0], color="green", ls="--", lw=1)
# ax.axhline(WORKABLE_BAND[1], color="green", ls="--", lw=1)
# ax.set_xlabel("input angle θ_in (deg)")
# ax.set_ylabel("transmission angle μ (deg)")
# ax.set_title("Transmission angle across the input range")
# ax.set_ylim(0, 180); ax.legend(); ax.grid(alpha=0.3)
# plt.tight_layout(); plt.show()

### ✏️ YOUR TURN — Transmission angle note

> _1–2 sentences. Does your linkage stay in the 40°–140° band throughout
> the input range? If not, name the input angle(s) where it leaves the
> band and what you'd change in your geometry to fix it. If it does, say
> so explicitly with the min/max μ values you observed._

---
## Section 5: Simulation and Interference Check [6 pts]

The third required analysis, plus the visible-motion evidence. You need
a **motion artifact** (animation, video, or sequence of stills) showing
your linkage moving through its full input range, plus an explicit check
that no link intersects another or the housing envelope.

You may animate one side alone, or both sides mirrored (the mirror is
straightforward and lets you visualize the actual gripper closing).

### Tool choices

- **The HTML starter** — `starters/four_bar_rapier_starter.html` is a
  single self-contained file you double-click. Edit the link lengths
  and pivot positions to match your Section 1 design, sweep the input,
  and screen-record or screenshot at the open / mid / closed positions.
- **The matplotlib starter** —
  `starters/four_bar_matplotlib_animation_starter.ipynb` is a Python
  notebook that produces the same animation inline. The last cell shows
  how to save MP4 or GIF directly into `motion/`.
- **Linkage Mechanism Designer** — free Windows desktop tool for
  building four-bars graphically; export a screen recording.
- **GeoGebra** — set up the linkage as constrained geometry, animate,
  export.
- **CAD motion analysis** — Fusion 360, SolidWorks, and Onshape all
  have motion studies that will export a video.

Whatever you use, save the artifact to `motion/` and reference it
below.

### ✏️ YOUR TURN — Motion artifact + interference check

**Tool used:** _e.g., "the HTML starter, edited to L1=…/L2=…/L3=…/L4=…
and OUT_X=…/OUT_Y=…, with mirrored other side enabled."_

**Motion artifact path:** `motion/<descriptive_filename>.{mp4,gif,zip,…}`
*(can be a video, animated GIF, a folder of stills, or a public link if
the file is too large to commit. If you link out, also commit a
representative still image.)*

_Embed a representative still by uncommenting:_
`<!-- ![Linkage motion](motion/four_bar_sweep_mid.png) -->`

**Interference check writeup (2–3 sentences):**
> _Did any link intersect another or the housing envelope at any point
> in the sweep? If yes, where, and how would you fix it? If no, name
> the closest approach you observed (e.g., "coupler passes within ~3 mm
> of the housing wall at θ_in = +45°")._

> **Forward connection.** In Part B, your team will overlay your
> motion artifact with three other linkages on the same axes (the
> Linkage Comparison Worksheet) — make sure your artifact clearly shows
> the input range you used so a teammate can replay it.

In [ ]:
# Quick check — what's in motion/?
print("Motion artifacts committed for Section 5:")
found = sorted(p for p in MOTION_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — save your animation/video/stills under motion/)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 6: Hand Calc at Three Positions [4 pts]

Anchor for the triangulation. Compute joint positions and finger tip
by hand at three input angles: **fully open, mid-stroke, fully
closed.** Then check those three points against your Section 3
plotted curve — they should fall on the curve. If they don't, one of
the two is wrong; you find out which.

**Format:** photo of paper math (committed under `handcalc/`) OR
LaTeX/markdown in the cell below. Either is acceptable. Mixed is fine.

**Pick whichever method you prefer:**

- Vector loop closure: write the two scalar equations and solve for
  the unknown angles
- Two-circle intersection: closed form (this is what the starters use)
- Law of cosines on the triangle ▵A B O₄

### ✏️ YOUR TURN — Hand calc

## Hand Calc — Position Analysis at Three Input Positions

**Approach:** _e.g., vector loop closure → two-circle intersection_

---

**At θ_in = ___° (fully open):**

- Working: _show work — can be photo of paper or LaTeX_
- Result: tip = ( _..._ , _..._ ) mm; single-side displacement from closed = _..._ mm
- Implied total jaw opening (= 2 × displacement): _..._ mm

**At θ_in = ___° (mid-stroke):**

- Working: _..._
- Result: tip = ( _..._ , _..._ ) mm; single-side displacement from closed = _..._ mm
- Implied total jaw opening: _..._ mm

**At θ_in = ___° (fully closed):**

- Working: _..._
- Result: tip = ( _..._ , _..._ ) mm; single-side displacement from closed = _..._ mm (should be ~0)

---

**Comparison to Section 3 plot:**
> _Do the three hand-calc points fall on your plotted displacement
> curve? Within what tolerance? If they don't, which one is wrong —
> the hand calc or the function — and how do you know?_

_If your hand calc is on paper, embed it by uncommenting:_
`<!-- ![Hand calc](handcalc/three_positions.jpg) -->`

In [ ]:
# Quick check — what's in handcalc/?
print("Hand calc artifacts committed for Section 6:")
found = sorted(p for p in HANDCALC_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — your hand calc may be entirely in markdown above, that's fine)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 7: Triangulation and Trust [4 pts]

Synthesis. This is where engineering judgment shows up most visibly.
Two artifacts:

1. **Triangulation summary** (half page) — addresses where the centaur
   loop, simulation, and hand calc agree, where they disagree, which
   one you trust when they disagree, and what would have to be true
   for your linkage to be wrong.
2. **Trust ledger** — what you trust about your linkage vs. what
   still needs checking before this becomes a printed part.

Specificity is graded. *"The linkage seems to work"* is not a trust
ledger entry. *"Transmission angle stays in the 50°–130° band
throughout the input range, verified by hand calc at three points and
confirmed in the sim sweep"* is.

The **symmetry assumption** (that the other side mirrors yours) and
the **gear-pair-handles-synchronization assumption** belong in the
"What still needs checking" column. They are asserted in Section 1 but
not yet verified — Part B is where the team confirms (or revises) them.

### ✏️ YOUR TURN — Triangulation summary (half-page)

**Where do the three paths agree?**
> _Be specific. "All three give a single-side displacement of 19–21 mm
> at θ_in = 60°, implying ~38–42 mm jaw opening." Don't say "they
> roughly agree."_

**Where do they disagree?**
> _Name the gap. "Centaur loop predicted μ_min = 35° (out of band),
> but my Section 4 plot shows μ_min = 47°. That's an 11° spread at the
> critical position."_

**When they disagree, which one do you trust and why?**
> _Engineering judgment is graded here. Cite the assumption that drove
> the difference. "I trust my Section 4 plot more because the centaur
> loop confused which angle is μ — it was computing the angle at A, not
> at B."_

**What would have to be true for your linkage design to be wrong?**
> _Name the failure mode you'd worry about even with all three paths
> agreeing. "If the gear pair my team designs in Part B uses a
> different ratio than I assumed, my input angle range shifts and the
> transmission angle could leave the band — that's the coupling Part B
> has to check."_

### ✏️ YOUR TURN — Trust ledger

Specificity is graded. Aim for at least three entries on each side.

| What I trust about this linkage | What still needs checking before this becomes a printed part |
|---------------------------------|--------------------------------------------------------------|
| _e.g., transmission angle stays in 50°–130° across full input range, verified by Section 4 plot and three hand calc points_ | _e.g., the symmetry assumption — the other side will mirror this only if my team's gear pair (Part B) actually counter-rotates at the same rate_ |
| _..._                            | _..._                                                         |
| _..._                            | _..._                                                         |

> **Forward connection.** Your team will combine the trust ledgers
> from each member's Part A into the per-subsystem trust assessment
> in Part B. Write yours so a teammate can pick it up cold.

---
## Section 8: Reflection [2 pts]

One short reflection. 3–4 sentences. Be specific — name a moment,
not a generality.

### What's the durable lesson?

> _If next quarter you have access to better AI tools but a different
> mechanism design problem, what part of what you just did would still
> apply? MCP, this specific stack, GitHub Models — these will be
> replaced. What's the underlying skill?_

---

## Submission checklist

Before pushing your final commit, verify:

- [ ] Section 1 design summary names link lengths, pivot positions,
      input range, target displacement, and includes a labeled sketch
- [ ] Section 2 has three rounds, each with at least one evidence file
      in `evidence/`
- [ ] Section 3 `compute_finger_position()` works; both plots render
- [ ] Section 4 `compute_transmission_angle()` works; plot renders with
      the workable band marked; in-band note is filled in
- [ ] Section 5 has a motion artifact in `motion/` (or a public link
      plus a representative still) and the interference writeup
- [ ] Section 6 hand calc covers three positions with working shown
      AND a comparison note tying the points to the Section 3 plot
- [ ] Section 7 triangulation summary and trust ledger are filled with
      specific entries; symmetry assumption appears as a known unknown
- [ ] Section 8 reflection is filled in

> **Codespaces save warning.** Make sure the notebook is saved
> (`Ctrl+S`) and committed via Source Control before stopping your
> Codespace. Unstaged changes are lost when the container is deleted.

> **Connection to Part B.** Your design summary (Section 1) and trust
> ledger (Section 7) are the input artifacts your team uses on day
> one of Part B for the linkage comparison. Make them readable
> standalone — your teammates won't have this notebook open.